# Exact numeric solution (v3 solver)

`src/moran_process/numeric` solves the Moran process exactly on all 2^N mutant sets
(Gauss-Seidel, over-relaxed, Gray-code order). What changed from `numeric_solution_2.ipynb`:

* **One run gives everything**: `numeric-cli --file <graph> <r>` prints, per starting node,
  `prob` (fixation probability), `time` (mean steps to absorption) and `ctime` (mean steps to
  fixation, conditional on fixing). There is no more `prob` / `time` action.
* **Threads = cores you get**: the CLI uses `available_parallelism()`, which follows the CPU
  affinity LSF sets, so `bsub -n k` means k threads. Each job logs `nproc` to prove it.
* **Memory is 12 B x 2^N** (three f32 per state): N=24 is 200 MB, N=28 is 3 GB, N=31 is 26 GB.
* **Limits**: 8 <= N <= 63 (asserted). Nodes with out-degree 0 are rejected by the text format
  (directed star / directed lung leaves), and directed graphs are unverified.

Every cell is non-blocking: it loads results that exist, shows the LSF state of jobs in flight,
submits the missing ones, and raises until all are ready. Re-run the cell later.

In [ ]:
%cd /home/labs/pilpel/matanyaw/moran-process
import re, subprocess
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from moran_process import PopulationGraph
from moran_process.analysis.analysis_utils.theory import analytic_moran_fc_fixation_prob

CRATE = Path("src/moran_process/numeric").resolve()
CLI = CRATE / "target/release/numeric-cli"
OUT = Path("simulation_data/numeric_solutions").resolve()
for d in ("graphs", "results", "logs"):
    (OUT / d).mkdir(parents=True, exist_ok=True)

# Incremental, a no-op when the binary is newer than the Rust sources (~7 s otherwise).
subprocess.run([Path.home() / ".cargo/bin/cargo", "build", "--release", "--bin", "numeric-cli"], cwd=CRATE, check=True)

In [ ]:
def job_name(g, r, cores):
    return f"{g.name}_r{r}_c{cores}"


def submit(g, r, cores, queue):
    name = job_name(g, r, cores)
    graph_file, out = OUT / "graphs" / f"{g.name}.txt", OUT / "results" / f"{name}.txt"
    g.to_simulation_struct().save(graph_file)
    mem_mb = 12 * 2**g.n_nodes / 2**20 * 1.2 + 512  # 3 f32 per state, plus headroom
    script = (f"echo start $(date +%s.%N) nproc $(nproc)\n"
              f"{CLI} --file {graph_file} {r} > {out}.tmp && mv {out}.tmp {out}\n"
              f"echo end $(date +%s.%N)\n")
    # Script on stdin: the WEXAC job starter mangles $vars and quotes on the bsub line.
    subprocess.run(["bsub", "-q", queue, "-J", name, "-n", str(cores),
                    "-R", f"span[hosts=1] rusage[mem={int(mem_mb / cores) + 1}]",  # per core
                    "-oo", str(OUT / "logs" / f"{name}.log")],
                   input=script, text=True, check=True, capture_output=True)


def solve(graphs, r, cores=4, queue="short"):
    # Returns {graph_name: DataFrame[prob, time, ctime] indexed by node}; raises until all ready.
    results, waiting = {}, []
    for g in graphs:
        name = job_name(g, r, cores)
        out, log = OUT / "results" / f"{name}.txt", OUT / "logs" / f"{name}.log"
        if out.exists():
            # Steps are integers, so a mean step count needs one decimal at most; the digits
            # beyond that are solver convergence noise (~1e-5 relative). Files keep full precision.
            results[g.name] = pd.read_csv(out, sep=r"\s+", nrows=g.n_nodes).round({"time": 1, "ctime": 1})
            continue
        state = subprocess.run(["bjobs", "-noheader", "-o", "stat", "-J", name],
                               capture_output=True, text=True).stdout.strip()
        if not state and log.exists() and "Exited with exit code" in log.read_text():
            state = f"FAILED (see {log.name}; delete it to retry)"
        elif not state:
            submit(g, r, cores, queue)
            state = "submitted"
        waiting.append(f"  {name:40} {state}")
    print(f"{len(results)}/{len(graphs)} ready", *waiting, sep="\n")
    if waiting:
        raise RuntimeError("not all results are ready: run this cell again later")
    return results


def wall_seconds(g, r, cores):
    # (seconds, nproc) of a finished job, from the timestamps its script logged.
    log = (OUT / "logs" / f"{job_name(g, r, cores)}.log").read_text()
    start, nproc = re.search(r"^start (\S+) nproc (\d+)", log, re.M).groups()
    end = re.search(r"^end (\S+)", log, re.M).group(1)
    return float(end) - float(start), int(nproc)


FIG_DIR = Path("figures/numeric_solution").resolve()
FIG_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams["svg.fonttype"] = "none"  # keep SVG text as text, so typst renders it in its own font


def draw(g, res, r, node_size=300):
    # One figure per quantity, each shown and saved as FIG_DIR/<graph>_r<r>_<kind>.svg.
    # prob is centered on the well-mixed value, so green = amplifies there.
    rho_wm = float(analytic_moran_fc_fixation_prob(g.n_nodes, r))
    panels = {
        "prob": dict(cmap="RdYlGn", label="fixation probability", center=rho_wm, vmin_floor=0.0,
                     half_width=max(np.abs(res["prob"] - rho_wm).max(), 0.5 * rho_wm)),
        "time": dict(label="mean steps to absorption"),
        "ctime": dict(label="mean steps to fixation"),
    }
    for kind, kw in panels.items():
        fig, ax = plt.subplots(figsize=(5.5, 4.8))
        g.draw_colored_graph(res[kind], ax=ax, node_size=node_size, **kw)
        fmt = ".4g" if kind == "prob" else ".1f"
        ax.set_title(f"{g.name}  (N={g.n_nodes}, r={r})   mean {res[kind].mean():{fmt}}")
        fig.savefig(FIG_DIR / f"{g.name}_r{r}_{kind}.svg", bbox_inches="tight")
        plt.show()


## 1. Solve and draw

Edit `GRAPHS`, `R`, `CORES`, `QUEUE` and run. Up to N ~ 21 a job takes seconds; each extra
node is ~2.25x slower and 2x the memory. Use `medium`/`long` for N >= 28.

In [ ]:
R, CORES, QUEUE = 1.1, 4, "short"
GRAPHS = [
    PopulationGraph.star_graph(16),
    PopulationGraph.cycle_graph(16),
    PopulationGraph.mammalian_lung_graph(branching_factor=2, depth=3),
    PopulationGraph.avian_graph(n_rods=6, rod_length=3),
    PopulationGraph.fish_graph(n_rods=3, rod_length=3, fillaments=2),
    PopulationGraph.grid_graph(4, 5),
]
results = solve(GRAPHS, R, CORES, QUEUE)

In [ ]:
for g in GRAPHS:
    draw(g, results[g.name], R)

## 2. Benchmark: wall time vs cores

The first cell submits one bsub job per core count, all at once. Each job runs the CLI on the
same graph, and the CLI uses every core LSF gives it, so `-n` is the only knob. The second
cell reads `Run time` from each job's LSF log (1 s resolution, fine for solves of minutes).
Use N >= 22: below that a solve takes under a second.

In [ ]:
from moran_process import PopulationGraph
from moran_process.numeric import compute
import subprocess
BENCH_CORES = [1, 2, 4, 8, 16]
BENCH_QUEUE = "short"


compute(PopulationGraph.line_graph(15).to_simulation_struct(), 1.1, 1)


In [ ]:
BENCH_G = PopulationGraph.line_graph(24)
graph_file = OUT / "graphs" / f"{BENCH_G.name}.txt"
BENCH_G.to_simulation_struct().save(graph_file)

for c in BENCH_CORES:
    name = f"bench_{BENCH_G.name}_c{c}"
    subprocess.run(f'bsub -q {BENCH_QUEUE} -n {c} -R "span[hosts=1] rusage[mem=1GB]" '
                   f'-J {name} -oo {OUT}/logs/{name}.log {CLI} --file {graph_file} 1.1',
                   shell=True, check=True)

In [ ]:
# Run after `bjobs` shows the jobs are done. A missing core count just means its job has not finished.
secs = {}
for c in BENCH_CORES:
    log = OUT / "logs" / f"bench_{BENCH_G.name}_c{c}.log"
    if log.exists() and (m := re.search(r"Run time :\s+(\d+) sec", log.read_text())):
        secs[c] = int(m.group(1))
df = pd.DataFrame({"seconds": secs})
df["speedup"] = df.seconds.iloc[0] / df.seconds
display(df)

plt.plot(df.index, df.speedup, "o-", label="measured")
plt.plot(df.index, df.index / df.index[0], "k--", lw=1, label="ideal")
plt.xlabel("cores (bsub -n)"); plt.ylabel(f"speedup vs {df.index[0]} core(s)")
plt.title(f"{BENCH_G.name} (N={BENCH_G.n_nodes}), r=1.1"); plt.legend(); plt.show()